# DataBaseHome

Extraction FBref pour les **5 dernières saisons** des Big 5 European Leagues.

## Reprise après interruption

Le script est **idempotent** : si tu le relances après un crash :
1. Les CSVs déjà écrits dans `XXYY/` sont **skippés** (sauf si `FORCE = True`).
2. Le **cache soccerdata** (`~/soccerdata/data/FBref`) est conservé : les pages HTML
   déjà téléchargées ne sont pas re-fetchées (gain massif).
3. Un **retry exponentiel** absorbe les erreurs transitoires (réseau, rate-limit).

Doc soccerdata : https://soccerdata.readthedocs.io/en/latest/

In [ ]:
import os
import time
import random
from pathlib import Path

import pandas as pd
import soccerdata as sd

pd.set_option('display.max_columns', None)
os.environ['SOCCERDATA_LOGLEVEL'] = 'ERROR'

# IMPORTANT : on LAISSE soccerdata cacher les pages HTML.
# Ne PAS définir SOCCERDATA_NOCACHE / SOCCERDATA_NOSTORE.

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────
SEASONS = ['2021', '2022', '2023', '2024', '2025']

# Vue agrégée FBref "Big 5 European Leagues Combined" : 1 seule requête HTML
# au lieu de 5 par stat_type → réduit drastiquement le risque de 403/rate-limit.
# Utilisée pour les stats SAISON (team + player) — la vue Combined supporte
# tous les stat_types (standard, keeper, keeper_adv, shooting, passing,
# passing_types, goal_shot_creation, defense, possession, playing_time, misc).
#
# Pour les stats par MATCH, le calendrier, lineups et events, la vue Combined
# n'expose pas les matchs individuels → on bascule sur la liste des 5 ligues.
LEAGUES_COMBINED = 'Big 5 European Leagues Combined'
LEAGUES_PER_COUNTRY = [
    'ENG-Premier League',
    'ESP-La Liga',
    'GER-Bundesliga',
    'ITA-Serie A',
    'FRA-Ligue 1',
]

# Vérification de version : 1.8.8 = full stat_types ; 1.9.0 a coupé passing/defense/etc.
assert sd.__version__.startswith('1.8'), (
    f"soccerdata {sd.__version__} détecté — installer 1.8.8 : "
    'pip install "soccerdata==1.8.8"'
)

# True = ré-écrase les CSVs existants. False = saute ce qui est déjà là.
FORCE = False

# Politesse FBref + paramètres retry
SLEEP_BASE = 3        # secondes entre requêtes (jitter ajouté)
MAX_RETRIES = 4       # tentatives par appel
BACKOFF_BASE = 30     # délai initial (×2, ×4, …)

STAT_TYPES_TEAM_SEASON = [
    'standard', 'keeper', 'keeper_adv', 'shooting', 'passing', 'passing_types',
    'goal_shot_creation', 'defense', 'possession', 'playing_time', 'misc',
]
STAT_TYPES_TEAM_MATCH = [
    'schedule', 'keeper', 'shooting', 'passing', 'passing_types',
    'goal_shot_creation', 'defense', 'possession', 'misc',
]
STAT_TYPES_PLAYER_SEASON = [
    'standard', 'shooting', 'passing', 'passing_types',
    'goal_shot_creation', 'defense', 'possession', 'playing_time', 'misc',
    'keeper', 'keeper_adv',
]
STAT_TYPES_PLAYER_MATCH = [
    'summary', 'keepers', 'passing', 'passing_types', 'defense', 'possession', 'misc',
]

BASE_DIR = Path.cwd()


def season_folder(season: str) -> str:
    """FBref season '2024' → folder '2425'."""
    yy = int(season) % 100
    return f"{yy:02d}{(yy + 1) % 100:02d}"


def polite_sleep():
    """Pause + jitter pour éviter de marteler FBref."""
    time.sleep(SLEEP_BASE + random.uniform(0, 1.5))


def with_retry(fn, label: str):
    """Exécute fn() avec retry exponentiel. Lève la dernière exception."""
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            return fn()
        except KeyboardInterrupt:
            raise
        except Exception as e:
            msg = str(e)
            # Erreurs métier : pas de retry
            if 'should be in' in msg or 'Invalid argument' in msg or 'has no attribute' in msg:
                raise
            if attempt == MAX_RETRIES:
                raise
            wait = BACKOFF_BASE * (2 ** (attempt - 1))
            print(f"     ↻ {label}: tentative {attempt}/{MAX_RETRIES} — {msg[:120]} — pause {wait}s")
            time.sleep(wait)


def safe_save(df, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=True)
    print(f"  ✓ {path.name}  ({len(df)} lignes)")


def fetch_one(label: str, path: Path, fn) -> bool:
    """
    Récupère un dataset, l'écrit, retourne True si OK.
    Skip si le fichier existe et FORCE=False.
    """
    if path.exists() and not FORCE:
        print(f"  · {path.name}  (déjà présent — skip)")
        return True
    try:
        polite_sleep()
        df = with_retry(fn, label)
        safe_save(df, path)
        return True
    except KeyboardInterrupt:
        raise
    except Exception as e:
        print(f"  ✗ {label}: {str(e)[:200]}")
        return False


def fetch_for_season(season: str, include_match_level: bool = True) -> None:
    """
    include_match_level=False saute les stats par match (très long).
    Utile pour un premier passage rapide saison-par-saison.
    """
    folder = BASE_DIR / season_folder(season)
    folder.mkdir(parents=True, exist_ok=True)
    print(f"\n=== Saison {season} → {folder.name}/ ===")

    # Instance "Combined" pour les stats saison (1 requête au lieu de 5).
    fb_combined = sd.FBref(leagues=LEAGUES_COMBINED, seasons=season)
    # Instance "par pays" pour le match-level (Combined n'a pas les matchs individuels).
    fb_per_country = sd.FBref(leagues=LEAGUES_PER_COUNTRY, seasons=season)

    # 1) Stats équipe — saison (vue Combined)
    for stat in STAT_TYPES_TEAM_SEASON:
        fetch_one(
            f'team_season_{stat}',
            folder / f'FB_team_season_stats_{stat}.csv',
            lambda s=stat: fb_combined.read_team_season_stats(stat_type=s),
        )

    # 2) Stats joueur — saison (vue Combined)
    for stat in STAT_TYPES_PLAYER_SEASON:
        fetch_one(
            f'player_season_{stat}',
            folder / f'FB_player_season_stats_{stat}.csv',
            lambda s=stat: fb_combined.read_player_season_stats(stat_type=s),
        )

    # 3) Calendrier + événements de tirs (par pays — Combined n'expose pas les matchs)
    fetch_one('schedule',    folder / 'FB_schedule.csv',    fb_per_country.read_schedule)
    fetch_one('shot_events', folder / 'FB_shot_events.csv', fb_per_country.read_shot_events)

    # 4) Stats par match — TRÈS LONG (des milliers de pages match) — par pays
    if include_match_level:
        for stat in STAT_TYPES_TEAM_MATCH:
            fetch_one(
                f'team_match_{stat}',
                folder / f'FB_team_match_stats_{stat}.csv',
                lambda s=stat: fb_per_country.read_team_match_stats(stat_type=s),
            )
        for stat in STAT_TYPES_PLAYER_MATCH:
            fetch_one(
                f'player_match_{stat}',
                folder / f'FB_player_match_stats_{stat}.csv',
                lambda s=stat: fb_per_country.read_player_match_stats(stat_type=s),
            )
        # Lineups + events : nécessitent que team_match_stats ait été appelé avant
        # (cache de la liste des matchs).
        for name, fn in [
            ('lineups', fb_per_country.read_lineup),
            ('events',  fb_per_country.read_events),
        ]:
            fetch_one(name, folder / f'FB_{name}.csv', fn)


## Phase 1 — Stats saison uniquement (rapide, ~10 min/saison)

À lancer en premier. Permet d'avoir l'app fonctionnelle vite, sans attendre les stats par match.

In [ ]:
for season in SEASONS:
    fetch_for_season(season, include_match_level=False)
print('\n✅ Phase 1 terminée.')

## Phase 2 — Stats par match (très long, plusieurs heures par saison)

À lancer ensuite. **Si interrompu, relance la cellule** : les fichiers déjà écrits seront skippés et le cache soccerdata reprend là où il s'est arrêté.

In [ ]:
for season in SEASONS:
    fetch_for_season(season, include_match_level=True)
print('\n✅ Extraction complète terminée.')

## Refetch ciblé

Pour rafraîchir une saison particulière (ex. saison en cours) :

In [ ]:
# Mets FORCE=True dans la cellule de config pour ré-écraser, puis :
fetch_for_season('2025', include_match_level=False)

## Diagnostic

Liste ce qui est présent / manquant pour chaque saison.

In [ ]:
"""
📊 Tableau de couverture — visualisation matricielle.
Affiche pour chaque saison × type de fichier : ✓ présent (taille MB) ou ✗ manquant.
"""
from IPython.display import display

CATEGORIES = {
    "Stats équipe (saison)":     [f"FB_team_season_stats_{s}.csv"   for s in STAT_TYPES_TEAM_SEASON],
    "Stats joueur (saison)":     [f"FB_player_season_stats_{s}.csv" for s in STAT_TYPES_PLAYER_SEASON],
    "Stats équipe (par match)":  [f"FB_team_match_stats_{s}.csv"    for s in STAT_TYPES_TEAM_MATCH],
    "Stats joueur (par match)":  [f"FB_player_match_stats_{s}.csv"  for s in STAT_TYPES_PLAYER_MATCH],
    "Calendrier & événements":   ["FB_schedule.csv", "FB_lineups.csv", "FB_events.csv"],
}

def file_info(path: Path):
    if not path.exists():
        return None
    size_mb = path.stat().st_size / 1e6
    try:
        with open(path, 'rb') as f:
            n = sum(1 for _ in f) - 1
    except Exception:
        n = -1
    return size_mb, n

# ── Stats globales ───────────────────────────────────────────────────────────
total_expected = 0
total_present  = 0
total_size_mb  = 0.0
total_rows     = 0
for season in SEASONS:
    folder_name = season_folder(season)
    for files in CATEGORIES.values():
        for f in files:
            total_expected += 1
            info = file_info(BASE_DIR / folder_name / f)
            if info:
                total_present += 1
                total_size_mb += info[0]
                total_rows    += max(info[1], 0)

pct = 100 * total_present / total_expected if total_expected else 0
print(f"📦 Couverture globale : {total_present}/{total_expected} fichiers ({pct:.1f}%)")
print(f"💾 Taille totale       : {total_size_mb:.1f} MB")
print(f"📊 Lignes totales      : {total_rows:,}".replace(',', ' '))
print(f"🧰 soccerdata version  : {sd.__version__}")


# ── Matrice par catégorie (tableau coloré) ───────────────────────────────────
def color_cell(val):
    if not isinstance(val, str): return ""
    if val.startswith("✓"): return "background-color:#1a4d2e;color:#fff"
    if val == "✗":          return "background-color:#5c1a1a;color:#fff"
    return ""

season_cols = [season_folder(s) for s in SEASONS]

for cat_name, files in CATEGORIES.items():
    print(f"\n━━━ {cat_name} ━━━")
    rows = []
    for f in files:
        stat_label = (
            f.replace("FB_team_season_stats_", "")
             .replace("FB_player_season_stats_", "")
             .replace("FB_team_match_stats_", "")
             .replace("FB_player_match_stats_", "")
             .replace("FB_", "").replace(".csv", "")
        )
        row = {"Fichier": stat_label}
        for season in SEASONS:
            folder_name = season_folder(season)
            info = file_info(BASE_DIR / folder_name / f)
            row[folder_name] = f"✓ {info[0]:.1f}MB" if info else "✗"
        rows.append(row)
    df_cov = pd.DataFrame(rows)
    styled = df_cov.style.applymap(color_cell, subset=season_cols)
    display(styled)


# ── Récap méthodes FBref ─────────────────────────────────────────────────────
print(f"\n━━━ Méthodes FBref (soccerdata {sd.__version__}) ━━━")
print("  read_team_season_stats   : standard, keeper, keeper_adv, shooting, passing,")
print("                             passing_types, goal_shot_creation, defense,")
print("                             possession, playing_time, misc")
print("  read_team_match_stats    : schedule, shooting, keeper, passing, passing_types,")
print("                             goal_shot_creation, defense, possession, misc")
print("  read_player_season_stats : (même set que team_season_stats)")
print("  read_player_match_stats  : summary, keepers, passing, passing_types, defense,")
print("                             possession, misc")
print("  read_schedule, read_lineup, read_events  (pas de stat_type)")
